# M3L4 E07 — Dashboard local de métricas
### Módulo 3 · Lecture 4 · Construcción, pruebas y trazabilidad de agentes en producción

---

## Qué necesitas saber antes

| Módulo | Concepto | Por qué lo necesitas acá |
|---|---|---|
| M3L4 E04-E06 | Routing accuracy, quality score, evaluación | Este notebook unifica todas esas métricas en un dashboard |
| M3L4 E03 | Latencia alta como patrón de falla | `p95_latency` es la métrica para detectar latencia alta |
| Python pandas | `groupby`, `mean`, `quantile`, filtros | Para calcular todas las métricas del dashboard |
| Python pandas | `nlargest()`, `sort_values()` | Para identificar los peores casos |

---

## Definiciones clave

| Concepto | Definición simple | Cómo aparece en este notebook |
|---|---|---|
| **Dashboard** | Conjunto de métricas que dan visibilidad del estado del sistema | DataFrame resumen con routing_accuracy, avg_latency, etc. |
| **routing_accuracy** | Proporción de consultas con intent correcto | `df['correct'].mean()` |
| **error_rate** | Proporción de consultas con intent incorrecto | `1 - routing_accuracy` o `(df['correct'] == 0).mean()` |
| **fallback_rate** | Proporción de consultas que caen en `general` | `(df['actual_intent'] == 'general').mean()` |
| **p95_latency** | El 95% de las requests tardan menos que este valor | `df['latency_ms'].quantile(0.95)` |
| **avg_quality_score** | Score de calidad promedio de respuestas | `df['quality_score'].mean()` |
| **Insight** | Conclusión accionable basada en las métricas | Texto que explica qué mejorar y cómo |

---

## Cómo encaja esto en un sistema de agentes

```
E04: Golden dataset -> routing_accuracy
E05: Comparación v1 vs v2 -> mejora de accuracy
E06: Evaluator agent -> quality_score
    |
    v
E07: Dashboard unificado (ESTE EJERCICIO)
    |  Todas las métricas en un solo lugar
    |  Identificar patrones y outliers
    v
E11: Golden dataset scores (métricas en el tiempo)
E12: Ciclo de mejora basado en insights del dashboard
```

**Objetivo del ejercicio:** analizar resultados de ejecuciones de agentes con pandas para obtener insights accionables.

## Instalación e imports

| Import | Qué hace | Por qué lo necesitamos |
|---|---|---|
| `pandas` (via pip) | DataFrames para análisis de métricas | Para cargar, filtrar y agregar datos de ejecuciones |
| `pd.DataFrame.quantile()` | Calcula percentiles | Para `p95_latency` |
| `pd.DataFrame.groupby()` | Agrupa por columna categórica | Para accuracy y latencia por intent |
| `json` | (Opcional) Para cargar traces desde archivo | Para integrar con datos reales en lugar de simulados |

```python
!pip install pandas -q
import pandas as pd
import json
```

In [ ]:
!pip install pandas -q
import pandas as pd
import json
print('pandas listo.')

## Dataset de ejecuciones (simulado)

Datos simulados de 15 ejecuciones del sistema multi-agente, cada una con:

| Columna | Descripción |
|---|---|
| `query` | Consulta del usuario |
| `expected_intent` | Intent esperado (ground truth) |
| `actual_intent` | Intent que devolvió el router |
| `latency_ms` | Duración total en milisegundos |
| `quality_score` | Score de calidad de la respuesta (0.0 a 1.0) |
| `correct` | 1 si expected == actual, 0 si no |

**Nota:** estos datos son simulados pero reflejan patrones reales: finance tiene fallos, IT tiene latencia alta, etc.

In [ ]:
execution_log = [
    {'query': 'Cómo solicito vacaciones?',                  'expected_intent': 'hr',          'actual_intent': 'hr',      'latency_ms': 800,  'quality_score': 0.9},
    {'query': 'No puedo ver mi factura',                     'expected_intent': 'finance',     'actual_intent': 'general', 'latency_ms': 1200, 'quality_score': 0.4},
    {'query': 'Mi VPN no conecta',                           'expected_intent': 'it',          'actual_intent': 'it',      'latency_ms': 950,  'quality_score': 0.8},
    {'query': 'Necesito el contrato actualizado',            'expected_intent': 'legal',       'actual_intent': 'legal',   'latency_ms': 1100, 'quality_score': 0.85},
    {'query': 'Cuándo se procesa el reembolso?',            'expected_intent': 'finance',     'actual_intent': 'general', 'latency_ms': 750,  'quality_score': 0.3},
    {'query': 'Mi laptop está lenta',                        'expected_intent': 'it',          'actual_intent': 'it',      'latency_ms': 4500, 'quality_score': 0.7},
    {'query': 'Quiero pedir licencia por maternidad',        'expected_intent': 'hr',          'actual_intent': 'hr',      'latency_ms': 880,  'quality_score': 0.95},
    {'query': 'Necesito firmar un NDA',                      'expected_intent': 'legal',       'actual_intent': 'legal',   'latency_ms': 1050, 'quality_score': 0.88},
    {'query': 'El sistema de login falla',                   'expected_intent': 'it',          'actual_intent': 'it',      'latency_ms': 920,  'quality_score': 0.75},
    {'query': 'ayuda',                                       'expected_intent': 'clarification','actual_intent': 'clarification', 'latency_ms': 200, 'quality_score': 1.0},
    {'query': 'Puedo ver mi comprobante de salario?',       'expected_intent': 'finance',     'actual_intent': 'hr',      'latency_ms': 850,  'quality_score': 0.5},
    {'query': 'No puedo entrar al portal rrhh',             'expected_intent': 'hr',          'actual_intent': 'hr',      'latency_ms': 780,  'quality_score': 0.92},
    {'query': 'El wifi de la oficina no funciona',           'expected_intent': 'it',          'actual_intent': 'it',      'latency_ms': 5000, 'quality_score': 0.65},
    {'query': 'Quiero hacer un reclamo',                     'expected_intent': 'finance',     'actual_intent': 'general', 'latency_ms': 700,  'quality_score': 0.35},
    {'query': 'Tengo un problema con mi laptop y mi factura','expected_intent': 'multi_intent','actual_intent': 'it',      'latency_ms': 1800, 'quality_score': 0.45},
]

df = pd.DataFrame(execution_log)
df['correct'] = (df['expected_intent'] == df['actual_intent']).astype(int)
print(f'Dataset: {len(df)} ejecuciones')
df.head()

## TODO 1 — Métricas globales

Calcula las métricas de resumen del sistema:

| Métrica | Expresión pandas |
|---|---|
| `routing_accuracy` | `df['correct'].mean()` |
| `error_rate` | `1 - df['correct'].mean()` o `(df['correct'] == 0).mean()` |
| `fallback_rate` | `(df['actual_intent'] == 'general').mean()` |
| `avg_latency` | `df['latency_ms'].mean()` |
| `p95_latency` | `df['latency_ms'].quantile(0.95)` |
| `avg_quality` | `df['quality_score'].mean()` |

In [ ]:
# TODO: calcular cada métrica
routing_accuracy = None   # media de 'correct'
error_rate       = None   # 1 - routing_accuracy
fallback_rate    = None   # % donde actual_intent == 'general'
avg_latency      = None   # media de latency_ms
p95_latency      = None   # percentil 95 de latency_ms
avg_quality      = None   # media de quality_score

print(f'Routing accuracy:  {routing_accuracy:.2%}')
print(f'Error rate:        {error_rate:.2%}')
print(f'Fallback rate:     {fallback_rate:.2%}')
print(f'Avg latency:       {avg_latency:.0f} ms')
print(f'P95 latency:       {p95_latency:.0f} ms')
print(f'Avg quality score: {avg_quality:.2f}')

## TODO 2 — Accuracy por intent

No alcanza con la métrica global. Hay que desglosar por intent para encontrar los débiles.

```python
accuracy_by_intent = df.groupby('expected_intent')['correct'].mean().sort_values()
accuracy_by_intent
```

**Por qué importa:** un accuracy global del 70% puede esconder que Finance tiene 0% y HR tiene 100%.

In [ ]:
# TODO: usar groupby + mean sobre 'correct', ordenado ascendente
accuracy_by_intent = None
accuracy_by_intent

## TODO 3 — Latencia por intent

Identificar qué intents tienen los peores tiempos de respuesta.

| Métrica | Expresión |
|---|---|
| Latencia promedio por intent | `df.groupby('expected_intent')['latency_ms'].mean()` |
| Latencia máxima por intent | `df.groupby('expected_intent')['latency_ms'].max()` |

In [ ]:
# TODO: latencia promedio y máxima por intent
latency_by_intent = None
latency_by_intent

## TODO 4 — Casos problemáticos

Identificar los casos que requieren atención inmediata:

1. **Alta latencia:** `latency_ms > 3000` (como vimos en E03)
2. **Baja calidad:** `quality_score < 0.5`
3. **Top 3 intents con más errores:** contar fallos por intent

Estos son los candidatos a mejora en el ciclo E12.

In [ ]:
# TODO 4a: casos con alta latencia (> 3000 ms)
high_latency_cases = None
print(f'Casos con alta latencia: {len(high_latency_cases) if high_latency_cases is not None else "N/A"}')

# TODO 4b: casos con baja calidad (quality_score < 0.5)
low_quality_cases = None
print(f'Casos con baja calidad: {len(low_quality_cases) if low_quality_cases is not None else "N/A"}')

# TODO 4c: top 3 intents con más errores
most_errors = None
most_errors

## Insight template

Completa el análisis con los valores que calculaste:

> El sistema tiene una routing accuracy general de **X%**. El intent con peor performance es **Y** (accuracy: **Z%**). Los casos de **finance** fallan porque el router no reconoce términos como: [listar]. La latencia máxima fue de **N ms** en el intent **K**.

> **Acción recomendada:** [completar]

In [ ]:
assert routing_accuracy is not None
assert error_rate is not None
assert avg_latency is not None
assert p95_latency is not None
assert avg_quality is not None
print(f'Checks E07 OK — Routing accuracy: {routing_accuracy:.2%}')

## Errores comunes

| Error | Causa | Cómo detectarlo |
|---|---|---|
| No calcular `correct` antes de usarlo | `df['correct']` no existe | Las métricas globales dan error |
| Usar `mean()` sobre columna incorrecta | Calcular media de `latency_ms` como accuracy | Accuracy > 100% |
| No ordenar accuracy por intent | Los valores aparecen en orden alfabético | Difícil ver cuál intent es peor |
| Confundir `quantile(0.95)` con `max()` | P95 no es lo mismo que el valor máximo | P95 excluye outliers extremos, max los incluye |
| No filtrar `correct == 0` antes de contar errores | Contar todos los casos en vez de solo fallos | El conteo de errores incluye aciertos |

## Síntesis

### Métricas del dashboard

| Métrica | Qué mide | Cómo se calcula |
|---|---|---|
| routing_accuracy | % de queries con routing correcto | `df['correct'].mean()` |
| error_rate | % de queries con routing incorrecto | `1 - routing_accuracy` |
| fallback_rate | % que cayó en `general` | `(df['actual_intent'] == 'general').mean()` |
| avg_latency | Velocidad promedio del sistema | `df['latency_ms'].mean()` |
| p95_latency | Latencia del percentil 95 | `df['latency_ms'].quantile(0.95)` |
| avg_quality | Calidad promedio de respuestas | `df['quality_score'].mean()` |
| accuracy_by_intent | Accuracy separado por categoría | `groupby('expected_intent')['correct'].mean()` |

### De los datos a la acción

```
Métricas globales       -> Qué tan bien funciona el sistema en general
    |
    v
Desglose por intent    -> Qué categorías tienen problemas
    |
    v
Casos problemáticos    -> Qué queries específicas fallan
    |
    v
Insight + acción       -> Qué mejorar y cómo
```

### Relación con otros ejercicios

| Ejercicio | Conexión con E07 |
|---|---|
| **E04-E05** | Las métricas de routing del dashboard vienen de ahí |
| **E06** | El `quality_score` del dashboard viene del evaluator agent |
| **E11** | Golden dataset scores: evolución de estas métricas en el tiempo |
| **E12** | Ciclo de mejora: insights del dashboard -> acciones correctivas |